
# Energy Consumption Anomaly Detection System

## Project Pipeline

This notebook includes:

1. Data Loading  
2. Data Cleaning  
3. Feature Engineering  
4. Isolation Forest Model  
5. Local Outlier Factor (LOF)  
6. Robust Covariance Model  
7. Business Visualizations  
8. Cost Impact Analysis  
9. Business Insights  

---


## 1️⃣ Imports

In [ ]:

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EllipticEnvelope


## 2️⃣ Data Loading

In [ ]:

def load_data(filepath):
    df = pd.read_csv(filepath, na_values=['null'])
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    print("Dataset Shape:", df.shape)
    return df


## 3️⃣ Data Cleaning

In [ ]:

def clean_data(df):
    initial_rows = df.shape[0]
    df = df.drop_duplicates()
    df = df.ffill()
    df = df.dropna(subset=['timestamp'])
    print("Rows Removed:", initial_rows - df.shape[0])
    return df


## 4️⃣ Feature Engineering

In [ ]:

def create_features(df, building):
    df['hour'] = df['timestamp'].dt.hour
    df['dayofweek'] = df['timestamp'].dt.dayofweek
    df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)
    df['lag_1'] = df[building].shift(1)
    df['lag_24'] = df[building].shift(24)
    df['rolling_mean_24'] = df[building].rolling(24).mean()
    df['rolling_std_24'] = df[building].rolling(24).std()
    df = df.dropna(subset=[building])
    print("After Feature Engineering:", df.shape)
    return df


## 5️⃣ Isolation Forest Model

In [ ]:

def run_isolation_forest(df, building):
    data = df[['timestamp', building]].dropna()
    X = data[[building]]

    model = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
    model.fit(X)

    data['anomaly'] = model.predict(X)
    data['anomaly'] = data['anomaly'].map({-1:1, 1:0})

    print("Isolation Forest Anomalies:", data['anomaly'].sum())
    return data


## 6️⃣ Local Outlier Factor (LOF)

In [ ]:

def run_lof(df, building):
    data = df[['timestamp', building]].dropna()
    X = data[[building]]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
    y_pred = model.fit_predict(X_scaled)

    data['anomaly'] = (y_pred == -1).astype(int)

    print("LOF Anomalies:", data['anomaly'].sum())
    return data


## 7️⃣ Robust Covariance Model

In [ ]:

def run_robust(df, building):
    data = df[['timestamp', building]].dropna()
    X = data[[building]]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = EllipticEnvelope(contamination=0.01, random_state=42)
    model.fit(X_scaled)

    y_pred = model.predict(X_scaled)
    data['anomaly'] = (y_pred == -1).astype(int)

    print("Robust Covariance Anomalies:", data['anomaly'].sum())
    return data


## 8️⃣ Visualizations

In [ ]:

def plot_energy_trend(df, building):
    plt.figure()
    plt.plot(df['timestamp'], df[building])
    plt.title("Energy Trend")
    plt.xlabel("Time")
    plt.ylabel("Energy")
    plt.show()


In [ ]:

def plot_anomalies(df, building):
    normal = df[df['anomaly']==0]
    anomalies = df[df['anomaly']==1]

    plt.figure()
    plt.plot(normal['timestamp'], normal[building])
    plt.scatter(anomalies['timestamp'], anomalies[building])
    plt.title("Anomaly Detection")
    plt.xlabel("Time")
    plt.ylabel("Energy")
    plt.show()


In [ ]:

def plot_monthly_anomalies(df):
    anomaly_df = df[df['anomaly']==1].copy()
    anomaly_df['month'] = anomaly_df['timestamp'].dt.month
    monthly = anomaly_df.groupby('month').size()

    plt.figure()
    plt.bar(monthly.index, monthly.values)
    plt.title("Monthly Anomalies")
    plt.xlabel("Month")
    plt.ylabel("Count")
    plt.show()


In [ ]:

def plot_hourly_heatmap(df, building):
    df_copy = df.copy()
    df_copy['hour'] = df_copy['timestamp'].dt.hour
    df_copy['day'] = df_copy['timestamp'].dt.dayofweek

    pivot = df_copy.pivot_table(values=building, index='day', columns='hour', aggfunc='mean')

    plt.figure()
    plt.imshow(pivot)
    plt.title("Hourly Heatmap")
    plt.xlabel("Hour")
    plt.ylabel("Day")
    plt.show()


## 9️⃣ Cost Impact Analysis

In [ ]:

def plot_cost_impact(df, building):
    anomaly_df = df[df['anomaly']==1].copy()
    avg_kwh_cost = 0.12
    anomaly_df['cost'] = anomaly_df[building] * avg_kwh_cost
    monthly_cost = anomaly_df.groupby(anomaly_df['timestamp'].dt.month)['cost'].sum()

    plt.figure()
    plt.bar(monthly_cost.index, monthly_cost.values)
    plt.title("Monthly Anomaly Cost")
    plt.xlabel("Month")
    plt.ylabel("Cost")
    plt.show()



## 🔟 Business Insights

- Isolation Forest captures global anomalies efficiently.
- LOF detects local density-based deviations.
- Robust Covariance assumes Gaussian distribution and identifies statistical outliers.
- Monthly anomaly trends help identify seasonal inefficiencies.
- Cost impact estimation quantifies financial losses due to abnormal energy consumption.

These insights support proactive maintenance and energy optimization decisions.


## ▶️ Run Full Pipeline

In [ ]:

building_column = "Building_1"

df = load_data("your_dataset.csv")
df = clean_data(df)
df = create_features(df, building_column)

iso_results = run_isolation_forest(df, building_column)
lof_results = run_lof(df, building_column)
robust_results = run_robust(df, building_column)

plot_energy_trend(df, building_column)
plot_anomalies(iso_results, building_column)
plot_monthly_anomalies(iso_results)
plot_hourly_heatmap(df, building_column)
plot_cost_impact(iso_results, building_column)
